**Data Loading and Initial Inspection**

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv"
Bank_dataframe = pd.read_csv(url)

extra_dataframe= pd.read_csv("Bank_Churn_Modelling_extra_UPDATED_v2.csv")

print("Bank Dataset Preview:")
display(Bank_dataframe.head())

print("Extra Dataset Preview:")
display(extra_dataframe.head())


In [2]:
Bank_dataframe.shape
extra_dataframe.shape

(49, 13)

1. **Structure comparison**
   
   The bank churn dataset has 10000 rows and 14 columns, the 2nd dataset has fewer records with similar customer attributes. Both datasets are tabular,
   rows are represent individual customers and columns represent variables. The second dataset however, has missing values and misaligned columns
   which makes the data incosistent.
   
2. **Schema alignment**

    The two datasets have similar key columns and because of that, they are compatible for merging but due to the incosistent formatting of the second     dataset, the data needs to be cleaned before merging them.

3. **Data Format**

    For both datasets, each row represents a single customer and each column represents a unique attribute, therefore the datasets are wide format.

4. **Format influence on wrangling, merging and aggregation.**

   Both datasets are wide formatwhich means datawrangling would be easy to do and as mentioned before, merging is possible due to the similar keys
   that exist in the two datasets.

   The second dataset has duplicate records and missing values which makes it inconsistent, which means that should we attempt to find the aggregate      as the data is, the results will be inaccurate. The data needs to be cleaned before it can be wrangled.


**Consolidation**

In [3]:
print("Rows in Bank dataset:", Bank_dataframe.shape[0])
print("Rows in Extra dataset:", extra_dataframe.shape[0])
extra_dataframe_clean = extra_dataframe.drop_duplicates(subset='CustomerId').reset_index(drop=True)
removed_duplicates = extra_dataframe.shape[0] - extra_dataframe_clean.shape[0]
print("Duplicate rows removed from extra dataset:", removed_duplicates)

combined_dataframe = pd.concat([Bank_dataframe, extra_dataframe_clean], ignore_index=True)
print("Rows after combining:", combined_dataframe.shape[0])
duplicate_ids = combined_dataframe['CustomerId'].duplicated().sum()
print("Number of duplicated CustomerId entries in combined dataset:", duplicate_ids)

Rows in Bank dataset: 10000
Rows in Extra dataset: 49
Duplicate rows removed from extra dataset: 9
Rows after combining: 10040
Number of duplicated CustomerId entries in combined dataset: 2


**Approach and Justification:**  
    I cimbined the two datasets by making use of concatenation(`pd.concat`) because: Each row represents a single customer observation, it allows us
    to retain all the records from both datasets and "CustomerId" is the key identifier for customers;

**Rows Before Combining:**  
    Bank dataset: 10,000 rows  
    Extra dataset: 49 

**Rows After Combining:**  
    Combined dataset: 10040

**Potential Data Integrity Implications:**  
    Two of the  `CustomerId` entries appeared in both datasets which means I have duplicates.  
    There is overlapping columns which could contain conflicting values.  
    The missing or misaligned data from the extra dataset may introduce NaNs.  
    The data needs to be cleaned up and duplicates deleted before the data is analysed.

**Duplicate Analysis and Resolution**

In [4]:
print("Dataset size before duplicate treatment:", combined_dataframe.shape)
full_row_duplicates_count = combined_dataframe.duplicated().sum()
print("Number of exact row duplicates:", full_row_duplicates_count)

pk_duplicates_count = combined_dataframe.duplicated(subset=['CustomerId'], keep=False).sum()
print("Number of primary key (CustomerId) duplicates:", pk_duplicates_count)

exact_duplicates = combined_dataframe[combined_dataframe.duplicated()]
print("Exact duplicate rows preview:")
print(exact_duplicates.head())

pk_duplicates = combined_dataframe[combined_dataframe.duplicated(subset=['CustomerId'], keep=False) & ~combined_dataframe.duplicated()]
print("Primary key duplicates preview:")
print(pk_duplicates.head())


Dataset size before duplicate treatment: (10040, 13)
Number of exact row duplicates: 0
Number of primary key (CustomerId) duplicates: 4
Exact duplicate rows preview:
Empty DataFrame
Columns: [CustomerId, Surname, CreditScore, Geography, Gender, Age, Tenure, Balance, Num Of Products, Has Credit Card, Is Active Member, Estimated Salary, Churn]
Index: []
Primary key duplicates preview:
       CustomerId  Surname  CreditScore Geography Gender   Age  Tenure  \
2325     15612193     Hsia        762.0     Spain   Male  29.0      10   
3132     15619407  Buckley        615.0    France   Male  39.0       4   
10031    15612193    Smith        711.0   Germany   Male  85.0       7   
10034    15619407    Jones        755.0    France   Male  39.0       2   

         Balance  Num Of Products  Has Credit Card  Is Active Member  \
2325   115545.33                2                1                 0   
3132   133707.09                1                1                 1   
10031  163004.71           

**Resolution**

In [5]:
combined_no_exact = combined_dataframe.drop_duplicates()
combined_cleaned = combined_no_exact.drop_duplicates(subset=['CustomerId'], keep='first')
print("Dataset size after duplicate treatment:", combined_cleaned.shape)

Dataset size after duplicate treatment: (10038, 13)


**Exact row duplicates:**
    Removed completely to eliminate redundancy.
    
**Primary key duplicates:**
    Multiple entries with the same `CustomerId` but conflicting values were found.
  
**Resolution Strategy:**  
    Duplicates are removed.  
    For primary key duplicates, the first occurence was retained which means the assumption is that it is the most accurate.

**Dataset Size:**  
    Before duplicate treatment: (10040,13)
    After duplicate treatment: (10038, 13)  

This ensures one unique record per customer and removes redundant or conflicting data, making the dataset ready for analysis.

**Finding and Filling Missing values**

In [6]:
missing_values = combined_cleaned.isnull().sum()
missing_values = missing_values[missing_values > 0]
print("Columns with missing values and their counts:")
print(missing_values)

Columns with missing values and their counts:
CreditScore         1
Gender              1
Age                 3
Balance             3
Estimated Salary    1
dtype: int64


**Impact of missing data on analysis:**  
Columns with missing numerical values will cause inaccuracies and bias calculations, columns with missing categorical values may affect group-based analyses and ignoring missing values could lead to inaccurate summaries or errors in machine learning models.

In [9]:
numerical_cols = combined_cleaned.select_dtypes(include=['float64', 'int64']).columns
for col in numerical_cols:
    if combined_cleaned[col].isnull().sum() > 0:
        combined_cleaned[col].fillna(combined_cleaned[col].median(), inplace=True)
     
categorical_cols = combined_cleaned.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if combined_cleaned[col].isnull().sum() > 0:
        combined_cleaned.loc[:, col] = combined_cleaned[col].fillna(...)
missing_after_treatment = combined_cleaned.isnull().sum()
print("Missing values after treatment:")
print(missing_after_treatment[missing_after_treatment > 0])

Missing values after treatment:
Series([], dtype: int64)


**Rationale for Treatment:** 

Numerical columns
    Median is used to highlight outliers and maintain central tendency without skewing the distribution.  
    
Categorical columns
    Mode keeps the most occuring category and avoids creating unknown categories.  
    
Verification
    The data does not contain any missing values, it is complete and it is ready for analysis and modelling.
This approach balances simplicity and data integrity, making the dataset reliable for subsequent analysis.

**Data Type and Structural refinements**

In [ ]:
combined_cleaned.dtypes
categorical_cols = ['Geography', 'Gender', 'IsActiveMember', 'Num Of Products']

In [ ]:
intended_categorical_cols = ['Geography', 'Gender', 'HasCreditCard', 'IsActiveMember', 'Num Of Products']
categorical_cols = [col for col in intended_categorical_cols if col in combined_cleaned.columns]

for col in categorical_cols:
    combined_cleaned.loc[:, col] = combined_cleaned[col].astype('category')

irrelevant_cols = [col for col in ['CustomerId', 'Surname'] if col in combined_cleaned.columns]
combined_cleaned = combined_cleaned.drop(columns=irrelevant_cols)

print("\nData types after conversion:")
print(combined_cleaned.dtypes)
print("\nRemaining columns:")
print(combined_cleaned.columns)

**Summary of Data Type and Structural Refinements:** 

 **Data types:**  
    Converted categorical variables (`Gender`, `Geography`, `HasCreditCard`, `IsActiveMember`, `Num Of Products`) to `category`.  
    Numerical variables (`Age`, `CreditScore`, `Balance`, `EstimatedSalary`, `Tenure`) remain as `int` or `float` for analysis.  

**Analytically irrelevant variables removed:**  
    CustomerId` and `Surname` are omited because they are identifiers and do not have analytical value.  

**Outcome:**  
    Dataset is now clean, memory-efficient, and ready for analysis or modelling.

**Customer Segmentation and Value-Based Structuring:**

In [11]:
import pandas as pd

combined_cleaned = combined_cleaned.copy()

bins = [0, 30, 45, 60, 100]

labels = ['Youth (18-30)', 'Young Adults (31-45)', 'Middle-aged (46-60)', 'Senior (61+)']

combined_cleaned['AgeGroup'] = pd.cut(combined_cleaned['Age'], bins=bins, labels=labels)

combined_cleaned[['Age', 'AgeGroup']].head()

,Age,AgeGroup
0,42.0,Young Adults (31-45)
1,41.0,Young Adults (31-45)
2,42.0,Young Adults (31-45)
3,39.0,Young Adults (31-45)
4,43.0,Young Adults (31-45)


**Segmentation Logic (Age-Based):**

The segmentation into age groups was according key life stages and typical financial behaviour patterns.

**Youth (18–30):** 
young people typically have less money and limited banking activity.
**Young Adults (31–45):** 
This age group typically has bit more income, more banking activity, and thus potential for more banking products.
**Middle-aged (46–60):** 
tends to be the peak in income, often with higher balances, investments, and definitely more product usage.
**Senior (61+):** 
older people are mostly in retirement and slightly different priorities like savings preservation and reduced risk exposure.

These categories were selected to:
Reflect financial behaviours across the different age groups, make meaningful comparison between customer groups and sipport targeted analysis and segmentation for the bank's business insights.

**Value-Based Segmentation**

In [32]:
import pandas as pd

combined_cleaned = combined_cleaned.copy()

value_bins = [0, 25000, 100000, 200000, combined_cleaned['Balance'].max()]
value_labels = ['Low', 'Medium', 'High', 'Very High']

combined_cleaned['ValueSegment'] = pd.cut(combined_cleaned['Balance'], bins=value_bins, labels=value_labels, include_lowest=True)

combined_cleaned[['Balance', 'ValueSegment']].head()

combined_cleaned['ValueSegment'].value_counts()


ValueSegment
High         4777
Low          3627
Medium       1589
Very High      42
Name: count, dtype: int64

**Value-Based Segmentation Logic:**

I segmented Customers according to the amount they have in their bank account using the `Balance` indicator. 
Four segments were defined:
Low: 0 – 25,000
Medium: 25,001 – 100,000
High: 100,001 – 200,000
Very High: 200,001+

I grouped customers based on their account balances to see who brings the most value to the bank. This helps the bank focus on high-value customers for things like targeted campaigns, loyalty rewards, or managing risk. We used `pd.cut()` to safely sort each customer into one of the value segments.

**Grouped Aggregation Analysis**

In [27]:
combined_cleaned = combined_cleaned.copy()

bins = [0, 30, 45, 60, 100]
labels = ['Youth', 'Young Adults', 'Middle-aged', 'Senior']

combined_cleaned['AgeGroup'] = pd.cut(combined_cleaned['Age'], bins=bins, labels=labels)

print("AgeGroup created")
combined_cleaned[['Age', 'AgeGroup']].head()

combined_cleaned.groupby('AgeGroup', observed=True)['Balance'].mean()

AgeGroup created


AgeGroup
Youth           73416.559584
Young Adults    76306.589082
Middle-aged     81702.206659
Senior          76299.786751
Name: Balance, dtype: float64

The analysis of the data shows that middle-aged customers on average, have the highest balance, suggesting they are in their peak earning years. Younger customers tend to have lower balances, which shows that they are still at developing stages of their careers.

**Pivot Table**

In [31]:
pivot_churn = pd.pivot_table(
    combined_cleaned,
    values='Churn',
    index='Geography',
    columns='Gender',
    aggfunc='mean'
)

pivot_churn

Gender,Female,Male
Geography,,
France,0.203622,0.127721
Germany,0.376872,0.278947
Spain,0.212980,0.132279


**What the pivot table shows:**
This pivot table shows the customer turnover rate aamongst different geographical locations and gender groups and the mean value shows the number of customers who left the bank in each group.

**Interpretation**
The results in the pivot  table can be used to compare churn behaviour amongst regions and genders which means that this helps identify the likeliness of certain groups of people to leave the bank and also, analysing the differences between male and female customers can help highlight behaviours even futher.